<a href="https://colab.research.google.com/github/yaranoun/ML-Tech/blob/main/notebooks/02_embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [22]:
!git clone https://github.com/yaranoun/ML-Tech.git

Cloning into 'ML-Tech'...
remote: Enumerating objects: 674, done.
remote: Counting objects: 100% (224/224), done.
remote: Compressing objects: 100% (208/208), done.
remote: Total 674 (delta 135), reused 32 (delta 16), pack-reused 450 (from 2)
Receiving objects: 100% (674/674), 703.85 KiB | 5.25 MiB/s, done.
Resolving deltas: 100% (410/410), done.


In [23]:
%cd /content/ML-Tech
!git pull origin main

/content/ML-Tech
From https://github.com/yaranoun/ML-Tech
 * branch            main       -> FETCH_HEAD
Already up to date.


In [24]:
import json

# each line of the bundled file is one {"user", "assistant"} chat example
with open("data/processed/chunks.json", "r", encoding="utf-8") as f:
    chunks = json.load(f)

print(f"loaded {len(chunks)} chunks")
print(chunks[0])

loaded 33 chunks
{'document': 'استمارة عن رخصة السوق العمومية لتقديمها للضمان الاجتماعي.txt', 'title': '"استمارة عن رخصة السوق العمومية لتقديمها للضمان الاجتماعي(يستوجب حضور صاحب العلاقة شخصيا)(للاطلاع على المستندات المطلوبة دون الحاجة لحجز موعد مسبق)"', 'url': 'https://tmo.gov.lb/web/panel/info/service-types/1', 'category': "Driver's Licence", 'service': 'رخصة سوق عمومية للضمان الاجتماعي', 'language': 'ar', 'keywords': "Driver's license, driving license, Lebanon, Lebanese, documents, appointment,taxi,cab, public, social security,رخصة سوق، لبنان، لبناني، المستندات ، عمومي، ضمان اجتماعي، موعد،", 'section': 'الوصف', 'text': 'لا يتطلب موعد\n(يستوجب حضور صاحب العلاقة شخصيا)\n1.1\nاستمارة تُطلب من قبل الصندوق الوطني للضمان الاجتماعي للمستفيدين من خدماته من فئة السائقين العموميين، وذلك للتأكّد من تجديدهم لرخصة السوق بصورة منتظمة.\n\n2'}


In [25]:
!pip install -q sentence-transformers faiss-cpu

In [26]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("intfloat/multilingual-e5-base")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

In [27]:
passages = [
    "passage: "
    + chunk["section"]
    + "\nKeywords: "
    + chunk.get("keywords", "")
    + "\n"
    + chunk["text"]
    for chunk in chunks
]

In [28]:
embeddings = embedding_model.encode(
    passages,
    normalize_embeddings=True,
    show_progress_bar=True
)

print(embeddings.shape)

Batches:   0%|          | 0/2 [00:00<?, ?it/s]

(33, 768)


In [29]:
import faiss

dimension = embeddings.shape[1]

index = faiss.IndexFlatIP(dimension)
index.add(embeddings)
faiss.write_index(index, "data/processed/passport_index.faiss")

In [30]:
faiss.write_index(
    index,
    "data/processed/passport_index.faiss"
)

In [31]:
import os

file_path = "data/processed/passport_index.faiss"
if os.path.exists(file_path):
    print(f"The file '{file_path}' exists.")
else:
    print(f"Error: The file '{file_path}' does not exist. Please re-run the cell that creates it (cell `CMhSIbmozWxF`).")

The file 'data/processed/passport_index.faiss' exists.


In [32]:
import faiss

# Load the index to confirm it was created successfully
loaded_index = faiss.read_index("data/processed/passport_index.faiss")
print(f"Loaded FAISS index with {loaded_index.ntotal} vectors and dimension {loaded_index.d}.")

Loaded FAISS index with 33 vectors and dimension 768.


In [33]:
def retrieve(question, k=3):
    query_embedding = embedding_model.encode(
        ["query: " + question],
        normalize_embeddings=True
    )

    scores, indices = index.search(query_embedding, k)

    results = []

    for score, idx in zip(scores[0], indices[0]):
        results.append({
            "score": float(score),
            "document": chunks[idx]["document"],
            "section": chunks[idx]["section"],
            "text": chunks[idx]["text"]
        })

    return results

In [34]:
results = retrieve("What are the fees to get a biometric passport?")

for result in results:
    print(result["score"])
    print(result["document"])
    print(result["section"])
    print(result["text"])
    print()

0.8698819875717163
Biometric Passport.txt
NB
To receive the passport immediately, the interested party can apply at the department of public relations. An additional fee of LBP 9,800,000 will be however requested.

0.8561791181564331
Biometric Passport.txt
Fees
Document requested	               Fees
Passport valid for 5 years	        6,000,000 L.L
Passport valid for 10 years	        10,000,000 L.L
Passport – first class - 5 years 	30,000,000 L.L
Passport – second class - 5 years 	20,000,000 L.L
Passport – first class - 10 years 	60,000,000 L.L
Passport – second class - 10 years 	40,000,000 L.L

0.8480343818664551
Biometric Passport.txt
Requested documents
The adequate application for passports format A4 (10 years) issued by the competent mayor according to the place of residence.
Lebanese ID card OR/AND an extract of civil status (whether the Lebanese citizen is applying for the 1st time for a biometric passport or not). Follow this link for more information: https://www.general-securi

In [35]:
!git config --global user.email "yarajnoun@gmail.com"
!git config --global user.name "yaranoun"

In [36]:
!git add data/processed/passport_index.faiss
!git commit -m "Update document FAISS"
!git pull --rebase origin main
!git push origin main

[main de27d4c] Update document FAISS
 1 file changed, 0 insertions(+), 0 deletions(-)
 rewrite data/processed/passport_index.faiss (98%)
From https://github.com/yaranoun/ML-Tech
 * branch            main       -> FETCH_HEAD
Current branch main is up to date.
fatal: could not read Username for 'https://github.com': No such device or address
